# Escalation cascade: filter → ML → LLM

"This looks suspicious, confirm suspicious, act accordingly." A cheap substring **filter**
screens everything; an **ML classifier** confirms only the filter hits; an **LLM** (Claude)
adjudicates only the ML-confirmed cases. Each tier exits early, so the expensive stages run
on a shrinking slice of traffic.

Driven directly via `org.agentic.flink.cascade.EscalationPipeline` (the streaming version is
`SuspiciousActivityCascadeExample`). Runs with no model server (built-in lexicon classifier);
set `ANTHROPIC_API_KEY` to enable the LLM tier.

Prereqs: `mvn -DskipTests package`, `pip install -e python/`.

In [1]:
import os, pathlib, subprocess
repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = [j for j in sorted((repo_root/'target').glob('agentic-flink-*.jar'))
            if 'original' not in j.name and 'sources' not in j.name]
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])
cp = repo_root/'target'/'runtime-classpath.txt'
if not cp.exists():
    subprocess.run(['mvn','-q','dependency:build-classpath',f'-Dmdep.outputFile={cp}',
                    '-Dmdep.includeScope=runtime'], cwd=repo_root, check=True)
import agentic_flink as af
af.start_jvm(extra_jars=[j for j in cp.read_text().strip().split(':') if j])
from agentic_flink import jclass

EP = jclass('org.agentic.flink.cascade.EscalationPipeline')
key = os.environ.get('ANTHROPIC_API_KEY')
b = EP.builder().withMlThreshold(0.5)
if key:
    b = b.withClaude(key)
pipeline = b.build()
print('cascade ready —', 'LLM tier: Claude' if key else 'LLM tier disabled (no ANTHROPIC_API_KEY)')

cascade ready — LLM tier disabled (no ANTHROPIC_API_KEY)


## Run a batch of messages through the cascade

In [2]:
messages = [
    'Hi team, lunch moved to 1pm in the usual room.',
    'Reminder: standup is at 9:30 tomorrow.',
    "Hello, I'd like to request a refund for order #4471.",
    'URGENT: your account will be suspended — verify your account now via this link.',
    'Please send a wire transfer and three gift cards immediately, this is time sensitive.',
    'Confirm your social security number and routing number to release your refund.',
    'The deployment finished and all checks are green.',
]
from collections import Counter
funnel = Counter()
for msg in messages:
    d = pipeline.evaluate(msg)
    tier = str(d.decidedBy); funnel[tier] += 1
    extra = f'  >> {d.llmRationale}' if tier == 'LLM' and d.llmRationale else ''
    print(f'{tier:<6} {d.verdict:<6} (ml={d.mlScore:.2f}) | {msg[:64]}{extra}')
print('\nFunnel — decided at each tier:', dict(funnel))

FILTER CLEAN  (ml=0.00) | Hi team, lunch moved to 1pm in the usual room.
FILTER CLEAN  (ml=0.00) | Reminder: standup is at 9:30 tomorrow.
ML     CLEAN  (ml=0.26) | Hello, I'd like to request a refund for order #4471.
ML     REVIEW (ml=0.80) | URGENT: your account will be suspended — verify your account now
ML     REVIEW (ml=0.78) | Please send a wire transfer and three gift cards immediately, th
ML     REVIEW (ml=0.82) | Confirm your social security number and routing number to releas
FILTER CLEAN  (ml=0.00) | The deployment finished and all checks are green.

Funnel — decided at each tier: {'FILTER': 3, 'ML': 4}


## What happened

- **FILTER / CLEAN** — no trigger term; never touched the ML model.
- **ML / CLEAN** — filter matched (e.g. "refund") but the weighted score stayed below threshold.
- **ML / REVIEW** — ML confirmed suspicious; with no LLM configured it routes to a human.
- **LLM / BLOCK|REVIEW|ALLOW** — only the ML-confirmed cases reach Claude, which decides the action.

Set `ANTHROPIC_API_KEY` and re-run cell 1 to turn the confirmed cases from `REVIEW` into real
Claude adjudications. Swap the lexicon classifier for a trained model by building with
`.withClassifier(new DjlInferenceConnection(...), setup)` — the `Classifier` SPI is identical.